# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process a FAIR² dataset via its Croissant schema using the `mlcroissant` library, referencing all dataset entities by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL for this dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}\n{metadata['description']}")

## 2. Data Overview

Explore available record sets and fields with their `@id`. This helps identify which parts of the dataset you may want to analyze.

In [ ]:
# List all record sets in the dataset by `@id` and name
print("Available Record Sets:")
for rec in dataset.metadata.record_sets:
    print(f"- @id: {rec['@id']} | name: {rec.get('name','(no name)')}")

# For each record set, print its fields (columns)
for rec in dataset.metadata.record_sets:
    print(f"\nFields in Record Set '@id': {rec['@id']}")
    for field in rec['fields']:
        print(f"  - @id: {field['@id']} | name: {field.get('name','(no name)')} | dataType: {field.get('dataType','(unknown)')}")

## 3. Data Extraction

Load data from the main record set into a DataFrame for analysis.

Replace `<record_set_id>` etc. below with the actual `@id` values from the overview step.

**In this dataset, the main survey record set is likely to be:**

- `@id`: `cr:RecordSet:survey_records` (example, check previous cell for the actual value)

You should always reference by `@id` for clarity and reliability.


In [ ]:
# Replace with the actual record set IDs from the previous cell
# For demonstration, we collect all record set `@id`s in a list
record_sets = [rec['@id'] for rec in dataset.metadata.record_sets]

dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records for record set: {record_set_id}")
    recs = list(dataset.records(record_set=record_set_id))
    if recs:
        dataframes[record_set_id] = pd.DataFrame(recs)
        print(f"Loaded {len(dataframes[record_set_id])} records for {record_set_id}")
    else:
        print(f"No records found for {record_set_id}")

# For analysis, pick the main survey record set ID by editing this variable if needed
main_record_set_id = record_sets[0] if record_sets else None
if main_record_set_id:
    print(f"\nColumns in main record set ({main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets available for extraction.")

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps such as filtering, normalization, handling missing values, or basic grouping.

Replace the field IDs below with an actual numeric field and a group field's `@id` from the field list previously printed. Example field IDs may include GAD-7/PHQ-9 total scores.

In [ ]:
# Example: Exploratory filtering and normalization
if main_record_set_id:
    df = dataframes[main_record_set_id]

    # Identify a numeric field by `@id` (use output from Section 2)
    # Example: 'gad7_score' or 'phq9_score' (actual @id may differ)
    # Let's demonstrate with the first numeric field found
    numeric_field = None
    group_field = None
    for rec in dataset.metadata.record_sets:
        if rec['@id'] == main_record_set_id:
            for field in rec['fields']:
                # Find a field with dataType Integer or Float
                if field.get('dataType') in ['schema:Integer', 'schema:Float', 'Integer', 'Float']:
                    numeric_field = field['@id']
                    break
            # Optionally select a group field (e.g. 'cr:Field:gender' or similar)
            for field in rec['fields']:
                if field.get('dataType') in ['schema:Text', 'Text', None] and field['@id'] != numeric_field:
                    group_field = field['@id']
                    break

    print(f"Using numeric field: {numeric_field}")
    print(f"Using group field: {group_field}")

    # Ensure these fields exist as DataFrame columns (mlcroissant exports columns by @id)
    if numeric_field and numeric_field in df.columns:
        # Remove missing or invalid data for numeric analysis
        df_num = pd.to_numeric(df[numeric_field], errors='coerce')
        mask = df_num > 10  # Example threshold
        filtered_df = df[mask].copy()
        print(f"Filtered records with {numeric_field} > 10:")
        display(filtered_df.head())

        # Normalization
        mean = df_num[mask].mean()
        std = df_num[mask].std()
        filtered_df[f"{numeric_field}_normalized"] = (df_num[mask] - mean) / std
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a categorical variable if present
        if group_field and group_field in df.columns:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped statistics by {group_field}:")
            display(grouped_df.head())
    else:
        print(f"No numeric field found for EDA in record set {main_record_set_id}.")

## 5. Visualization

Visualize distributions or relationships. You can use matplotlib or seaborn. Here we show a histogram and a boxplot of the main numeric field, optionally grouped.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna().astype(float), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Optional: boxplot by group, if group field exists
    if group_field and group_field in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field].astype(float))
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion

This notebook demonstrated how to load and analyze a FAIR² Croissant dataset using the `mlcroissant` library, referencing all structures and columns by their `@id`. You can adjust the code to focus on specific fields (using the IDs shown in Section 2) for your research and extend EDA as needed.

- All field and record set references are by their Croissant `@id`, ensuring reproducibility and schema-alignment.
- Proceed to further modeling or statistical analysis with the prepared and explored data!
